# Question 2: How Do the Two Campaigns Compare?
**Ohio 2024 U.S. Senate race: Sherrod Brown (Democrat) vs Bernie Moreno (Republican). FEC data, 2023-24 cycle.**

**Question:** How do Brown's and Moreno's campaigns compare in spending level, categories, communication, timing and vendors?

**What this notebook answers:** What each campaign spent, what it spent it on, how it communicated with voters, who it paid, and how its committees differ. Only the campaigns' own spending is used here. Outside groups are covered in Question 3.

## Setup and loading the data

In [1]:
# Imports and output folder
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = "../outputs/question_2/"
os.makedirs(OUT, exist_ok=True)
pd.options.display.float_format = "{:,.2f}".format

In [2]:
# Load the cleaned spending file. Campaign = the candidates' own committees; affiliated = joint fundraising and victory committees
spend = pd.read_csv("../data/processed/campaign_spending.csv", parse_dates=["date"])
print(len(spend), "rows loaded")

df = spend[spend["include"] == True].copy()
print(len(df), "rows have include == True")

df["who"] = df["candidate"].map({"Sherrod Brown": "Brown", "Bernie Moreno": "Moreno"})

campaign = df[df["group"] == "campaign"].copy()
affiliated = df[df["group"] == "affiliated"].copy()
print(len(campaign), "campaign payments;", len(affiliated), "affiliated payments (used only in metric 9)")

10718 rows loaded
5912 rows have include == True
4700 campaign payments; 1212 affiliated payments (used only in metric 9)


In [3]:
# Colours, communication categories and the shared chart function
BROWN, MORENO = "#1f5fa8", "#c0392b"
COLORS = {"Brown": BROWN, "Moreno": MORENO}
WHO = ["Brown", "Moreno"]

COMM = ["TV, radio & other media", "Digital advertising", "Direct mail & printing",
        "Texting & phone", "Canvassing & field"]
COMM_COLORS = dict(zip(COMM, ["#6a51a3", "#e6a817", "#2a9d8f", "#8d6e63", "#9e9e9e"]))

SCOPE = "Campaign committees only, 2023–2024"
total = campaign.groupby("who")["amount"].sum()
print(total.round(2))

FIGS = {}
CHARTS = []

def money_m(x):
    """Dollars -> text like $48.43M"""
    return f"${x / 1e6:,.2f}M"

def finish(fig, filename, title, scope, height=620, legend=True, top=150, legend_y=1.02, legend_right=False):
    """Apply the shared chart layout, save the chart as an HTML file and show it."""
    if legend_right:
        legend_layout = {"orientation": "v", "x": 1.01, "xanchor": "left", "y": 1, "yanchor": "top"}
        right = 240
    else:
        legend_layout = {"orientation": "h", "x": 1, "xanchor": "right", "y": legend_y, "yanchor": "bottom"}
        right = 40
    fig.update_layout(
        title={"text": f"<b>{title}</b><br><sup>{scope}</sup>", "x": 0.02, "y": 0.97, "yanchor": "top"},
        template="plotly_white", height=height, showlegend=legend, legend=legend_layout,
        margin={"t": top, "b": 100, "l": 80, "r": right}, font={"size": 13})
    fig.update_yaxes(automargin=True)
    fig.write_html(OUT + filename, include_plotlyjs=True)
    FIGS[filename] = fig
    CHARTS.append(filename)
    fig.show()

who
Brown    97,460,584.55
Moreno   25,820,920.56
Name: amount, dtype: float64


## Metric 1. Side-by-side scorecard
**What it measures:** the headline numbers for each campaign: how much it spent, how much of that went to voters, how many payments it made, a typical payment, its biggest payment and its biggest vendor.

In [4]:
# Scorecard numbers for each candidate
stats = {}
for w in WHO:
    g = campaign[campaign["who"] == w]
    vendors = g.groupby("payee")["amount"].sum().sort_values(ascending=False)
    comm_dollars = g[g["category"].isin(COMM)]["amount"].sum()
    stats[w] = {
        "total_spent": g["amount"].sum(),
        "communication_share_pct": comm_dollars / g["amount"].sum() * 100,
        "payments": len(g),
        "median_payment": g["amount"].median(),
        "biggest_payment": g["amount"].max(),
        "biggest_vendor": vendors.index[0],
        "biggest_vendor_share_pct": vendors.iloc[0] / g["amount"].sum() * 100,
    }
stats = pd.DataFrame(stats)

keys = ["total_spent", "communication_share_pct", "payments", "median_payment", "biggest_payment", "biggest_vendor_share_pct"]
labels = ["Total spent ($)", "Share spent on voter communication (%)", "Number of payments",
          "Typical payment: median ($)", "Biggest single payment ($)", "Biggest vendor's share of own spending (%)"]
scorecard = pd.DataFrame({"measure": labels,
                          "Brown": [float(stats.loc[k, "Brown"]) for k in keys],
                          "Moreno": [float(stats.loc[k, "Moreno"]) for k in keys]})

scorecard["Brown vs Moreno"] = scorecard["Brown"] / scorecard["Moreno"]
scorecard.loc[1, "Brown vs Moreno"] = scorecard.loc[1, "Brown"] - scorecard.loc[1, "Moreno"]
scorecard.loc[5, "Brown vs Moreno"] = np.nan
scorecard["how_compared"] = ["Brown ÷ Moreno (times)", "Brown minus Moreno (percentage points)", "Brown ÷ Moreno (times)",
                             "Brown ÷ Moreno (times)", "Brown ÷ Moreno (times)", "not compared"]
scorecard["biggest_vendor_Brown"] = ["", "", "", "", "", stats.loc["biggest_vendor", "Brown"]]
scorecard["biggest_vendor_Moreno"] = ["", "", "", "", "", stats.loc["biggest_vendor", "Moreno"]]
scorecard.to_csv(OUT + "01_scorecard.csv", index=False)
scorecard

,measure,Brown,Moreno,Brown vs Moreno,how_compared,biggest_vendor_Brown,biggest_vendor_Moreno
0,Total spent ($),"97,460,584.55","25,820,920.56",3.77,Brown ÷ Moreno (times),,
1,Share spent on voter communication (%),82.31,79.67,2.64,Brown minus Moreno (percentage points),,
2,Number of payments,"2,517.00","2,183.00",1.15,Brown ÷ Moreno (times),,
3,Typical payment: median ($),996.00,548.47,1.82,Brown ÷ Moreno (times),,
4,Biggest single payment ($),"2,652,813.00","2,128,576.63",1.25,Brown ÷ Moreno (times),,
5,Biggest vendor's share of own spending (%),48.12,62.23,NaN,not compared,GRASSROOTS MEDIA,STRATEGIC MEDIA PLACEMENT


In [5]:
# Chart 1: scorecard
short = ["Total spent", "Share spent on voter communication", "Number of payments", "Typical payment (median)", "Biggest single payment"]
scale = [1e6, 1, 1, 1, 1, 1]
card_format = [dict(prefix="$", suffix="M", valueformat=",.1f"), dict(suffix="%", valueformat=".1f"),
               dict(valueformat=",.0f"), dict(prefix="$", valueformat=",.0f"),
               dict(prefix="$", valueformat=",.0f"), dict(suffix="%", valueformat=".1f")]
ratio_format = [dict(suffix=" times", valueformat=".1f"), dict(prefix="+", suffix=" pts", valueformat=".1f"),
                dict(suffix=" times", valueformat=".2f"), dict(suffix=" times", valueformat=".2f"),
                dict(suffix=" times", valueformat=".2f")]

fig = make_subplots(rows=6, cols=3, specs=[[{"type": "indicator"}] * 3 for _ in range(6)], vertical_spacing=0.07,
                    column_titles=[f'<b><span style="color:{BROWN}">Brown</span></b>', f'<b><span style="color:{MORENO}">Moreno</span></b>', "<b>Brown vs Moreno</b>"])
for r in range(6):
    for c, w in enumerate(WHO):
        if r < 5:
            card_title = short[r]
        else:
            card_title = "Biggest vendor: " + stats.loc["biggest_vendor", w].title()
        fig.add_trace(go.Indicator(mode="number", value=scorecard.loc[r, w] / scale[r],
                                   number=dict(font=dict(size=34, color=COLORS[w]), **card_format[r]),
                                   title=dict(text=card_title, font=dict(size=13))), row=r + 1, col=c + 1)
    if r < 5:
        fig.add_trace(go.Indicator(mode="number", value=scorecard.loc[r, "Brown vs Moreno"],
                                   number=dict(font=dict(size=28, color="#333333"), **ratio_format[r]),
                                   title=dict(text=short[r], font=dict(size=13))), row=r + 1, col=3)
finish(fig, "01_scorecard.html", "Brown vs Moreno: the scorecard", SCOPE, height=1050, legend=False, top=150)

## Metric 2. What the money was spent on, by category
**What it measures:** dollars (and share of each candidate's own spending) in each of the 16 existing categories. Use the buttons to switch between **$** and **% of own spending**. All 16 categories are shown; none were grouped.

In [6]:
# Spending by category, in dollars and as a share of each candidate's total
cat = campaign.pivot_table(index="category", columns="who", values="amount", aggfunc="sum", fill_value=0)
cat = cat.reindex(columns=WHO, fill_value=0)
cat["combined"] = cat["Brown"] + cat["Moreno"]
cat = cat.sort_values("combined", ascending=False)
for w in WHO:
    cat[w + "_pct"] = cat[w] / total[w] * 100
category_table = cat.reset_index()
category_table.to_csv(OUT + "02_category_spending.csv", index=False)
category_table

who,category,Brown,Moreno,combined,Brown_pct,Moreno_pct
0,"TV, radio & other media","48,431,013.77","16,592,280.04","65,023,293.81",49.69,64.26
1,Digital advertising,"24,294,063.05","1,870,906.38","26,164,969.43",24.93,7.25
2,Payroll & benefits,"5,951,014.68","101,207.25","6,052,221.93",6.11,0.39
3,Direct mail & printing,"4,484,492.48","1,523,814.23","6,008,306.71",4.60,5.90
4,Fundraising costs,"4,356,828.10","1,131,935.57","5,488,763.67",4.47,4.38
5,Texting & phone,"2,933,060.27","561,475.30","3,494,535.57",3.01,2.17
6,Processing & bank fees,"2,689,479.63","363,734.88","3,053,214.51",2.76,1.41
7,Consulting & strategy,"817,941.75","1,644,271.97","2,462,213.72",0.84,6.37
8,Reimbursements & card payments,"914,113.00","1,066,159.67","1,980,272.67",0.94,4.13
9,Operations & overhead,"1,233,279.54","156,813.82","1,390,093.36",1.27,0.61


In [7]:
# Chart 2: spending by category ($ and % switch)
fig = go.Figure()
for mode in ["dollars", "percent"]:
    for w in WHO:
        if mode == "dollars":
            values = category_table[w] / 1e6
            name = w + " ($M)"
        else:
            values = category_table[w + "_pct"]
            name = w + " (% of own spending)"
        hover = [f"<b>{row['category']}</b><br>{w}: {money_m(row[w])} ({row[w + '_pct']:.1f}% of {w}'s spending)"
                 for _, row in category_table.iterrows()]
        fig.add_trace(go.Bar(y=category_table["category"], x=values, orientation="h", name=name,
                             marker_color=COLORS[w], hovertext=hover, hoverinfo="text",
                             visible=(mode == "dollars")))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title_text="Spent ($M)")
fig.update_layout(barmode="group", updatemenus=[dict(
    type="buttons", direction="right", x=0, xanchor="left", y=1.02, yanchor="bottom",
    buttons=[dict(label="$ spent", method="update",
                  args=[{"visible": [True, True, False, False]}, {"xaxis.title.text": "Spent ($M)"}]),
             dict(label="% of own spending", method="update",
                  args=[{"visible": [False, False, True, True]}, {"xaxis.title.text": "Share of own spending (%)"}])])])
finish(fig, "02_category_spending.html", "What each campaign spent its money on", SCOPE, height=780)

## Metric 3. Category comparison: who puts a bigger share where?
**What it measures:** for each category, Brown's share of his own spending (x) against Moreno's share of his own spending (y).

**How to read it:** a dot **above the dashed line** means Moreno puts a *bigger share* of his spending there; **below the line** means Brown does. The right-hand chart zooms in on the small categories so their labels do not overlap.

In [8]:
# Each category as a share of the candidate's own spending
shares = category_table[["category", "Brown_pct", "Moreno_pct"]].copy()
shares["puts_more_share_here"] = np.where(shares["Moreno_pct"] > shares["Brown_pct"], "Moreno", "Brown")
shares["biggest_share"] = shares[["Brown_pct", "Moreno_pct"]].max(axis=1)
shares["in_zoom_chart"] = shares["biggest_share"] < 8
shares["labelled_left"] = shares["biggest_share"] >= 20
shares["labelled_zoom"] = shares["in_zoom_chart"] & (shares["biggest_share"] >= 1)
shares.to_csv(OUT + "03_category_shares.csv", index=False)
shares

who,category,Brown_pct,Moreno_pct,puts_more_share_here,biggest_share,in_zoom_chart,labelled_left,labelled_zoom
0,"TV, radio & other media",49.69,64.26,Moreno,64.26,False,True,False
1,Digital advertising,24.93,7.25,Brown,24.93,False,True,False
2,Payroll & benefits,6.11,0.39,Brown,6.11,True,False,True
3,Direct mail & printing,4.60,5.90,Moreno,5.90,True,False,True
4,Fundraising costs,4.47,4.38,Brown,4.47,True,False,True
5,Texting & phone,3.01,2.17,Brown,3.01,True,False,True
6,Processing & bank fees,2.76,1.41,Brown,2.76,True,False,True
7,Consulting & strategy,0.84,6.37,Moreno,6.37,True,False,True
8,Reimbursements & card payments,0.94,4.13,Moreno,4.13,True,False,True
9,Operations & overhead,1.27,0.61,Brown,1.27,True,False,True


In [9]:
# Chart 3: Brown's share against Moreno's share for each category
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=["All categories (small ones are labelled in the zoom)", "Zoom: categories under 8% of either campaign"])

def add_dots(panel, subset, label_column, show_legend):
    """Draw the dots of one panel: blue = Brown puts more share there, red = Moreno puts more share there."""
    for w in WHO:
        part = subset[subset["puts_more_share_here"] == w]
        if label_column is None:
            text = [""] * len(part)
        else:
            text = [name if show else "" for name, show in zip(part["category"], part[label_column])]
        hover = [f"<b>{r['category']}</b><br>Brown: {r['Brown_pct']:.1f}% of his spending<br>Moreno: {r['Moreno_pct']:.1f}% of his spending"
                 for _, r in part.iterrows()]
        fig.add_trace(go.Scatter(x=part["Brown_pct"], y=part["Moreno_pct"], mode="markers+text", text=text,
                                 textposition=["middle left" if n == "TV, radio & other media" else "middle right" for n in part["category"]], marker=dict(size=11, color=COLORS[w]),
                                 hovertext=hover, hoverinfo="text", showlegend=show_legend,
                                 name=w + " puts a bigger share here"), row=1, col=panel)

add_dots(1, shares, "labelled_left", True)
add_dots(2, shares[shares["in_zoom_chart"]], None, False)

# Label positions for the zoom panel: text, pixels right, pixels down
zoom_labels = {
    "Consulting & strategy": ("Consulting & strategy", 30, 0),
    "Direct mail & printing": ("Direct mail & printing", 30, 0),
    "Fundraising costs": ("Fundraising costs", 35, 28),
    "Reimbursements & card payments": ("Reimbursements &<br>card payments", 30, -15),
    "Texting & phone": ("Texting & phone", 35, -20),
    "Legal & compliance": ("Legal &<br>compliance", 30, -22),
    "Travel, events & meals": ("Travel, events<br>& meals", 30, 8),
    "Processing & bank fees": ("Processing &<br>bank fees", 38, 12),
    "Operations & overhead": ("Operations & overhead", 35, 18),
    "Payroll & benefits": ("Payroll & benefits", 35, -15),
}
for _, r in shares[shares["labelled_zoom"]].iterrows():
    label, move_right, move_down = zoom_labels[r["category"]]
    fig.add_annotation(x=r["Brown_pct"], y=r["Moreno_pct"], xref="x2", yref="y2", text=label,
                       showarrow=True, arrowhead=0, arrowwidth=1, arrowcolor="#999999", ax=move_right, ay=move_down,
                       axref="pixel", ayref="pixel", xanchor="left", align="left",
                       bgcolor="rgba(255,255,255,0.85)", font=dict(size=12))
for panel, top in [(1, 70), (2, 7.5)]:
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", line=dict(dash="dash", color="#888888"),
                             name="Equal share", showlegend=(panel == 1), hoverinfo="skip"), row=1, col=panel)
    fig.update_xaxes(title_text="Brown: share of his own spending (%)", range=[-0.5, 10] if panel == 2 else [0, 70], row=1, col=panel)
    fig.update_yaxes(title_text="Moreno: share of his own spending (%)", range=[-0.5, 7.5] if panel == 2 else [0, 70], row=1, col=panel)
finish(fig, "03_category_scatter.html", "Which categories does each campaign favour?", SCOPE,
       height=780, top=170, legend_y=1.09)

## Metric 4. What makes up Brown's spending advantage?
**What it measures:** for each category, Brown's dollars minus Moreno's dollars. Blue bars = Brown spent more; red bars = Moreno spent more.

In [10]:
# Brown minus Moreno for each category
gap = category_table[["category", "Brown", "Moreno"]].copy()
gap["gap"] = gap["Brown"] - gap["Moreno"]
gap = gap.sort_values("gap", ascending=False)
total_gap = gap["gap"].sum()
gap["share_of_total_gap_pct"] = gap["gap"] / total_gap * 100
gap.to_csv(OUT + "04_brown_advantage.csv", index=False)
print("Total gap (Brown minus Moreno):", round(total_gap, 2))
gap

Total gap (Brown minus Moreno): 71639663.99


who,category,Brown,Moreno,gap,share_of_total_gap_pct
0,"TV, radio & other media","48,431,013.77","16,592,280.04","31,838,733.73",44.44
1,Digital advertising,"24,294,063.05","1,870,906.38","22,423,156.67",31.30
2,Payroll & benefits,"5,951,014.68","101,207.25","5,849,807.43",8.17
4,Fundraising costs,"4,356,828.10","1,131,935.57","3,224,892.53",4.50
3,Direct mail & printing,"4,484,492.48","1,523,814.23","2,960,678.25",4.13
5,Texting & phone,"2,933,060.27","561,475.30","2,371,584.97",3.31
6,Processing & bank fees,"2,689,479.63","363,734.88","2,325,744.75",3.25
9,Operations & overhead,"1,233,279.54","156,813.82","1,076,465.72",1.50
10,Legal & compliance,"660,555.33","436,607.57","223,947.76",0.31
11,"Travel, events & meals","440,026.67","295,222.02","144,804.65",0.20


In [11]:
# Chart 4: Brown's advantage by category
fig = go.Figure()
for w, label, mask in [("Brown", "Brown spent more", gap["gap"] >= 0), ("Moreno", "Moreno spent more", gap["gap"] < 0)]:
    part = gap[mask]
    hover = [f"<b>{r['category']}</b><br>Brown: {money_m(r['Brown'])}<br>Moreno: {money_m(r['Moreno'])}<br>Difference: {'+' if r['gap'] >= 0 else '-'}${abs(r['gap']) / 1e6:,.2f}M ({r['share_of_total_gap_pct']:.1f}% of total gap)"
             for _, r in part.iterrows()]
    fig.add_trace(go.Bar(y=part["category"], x=part["gap"] / 1e6, orientation="h", name=label, marker_color=COLORS[w],
                         text=[f"{v / 1e6:+.2f}" for v in part["gap"]], textposition="outside",
                         hovertext=hover, hoverinfo="text"))
fig.update_yaxes(autorange="reversed", categoryorder="array", categoryarray=list(gap["category"]))
fig.update_xaxes(title_text="Brown spent this much more than Moreno ($M)", range=[-4, 38])
finish(fig, "04_brown_advantage.html", f"Brown spent {money_m(total_gap)} more than Moreno in total", SCOPE, height=760, top=140)

## Metric 5. Types of communication
**What it measures:** how each candidate's voter-communication dollars were split across the existing communication categories. The donut centre shows the candidate's total communication spending.

*Note:* the existing categories have no separate "other communication" group, and we did not create one. "Canvassing & field" is the only field/event-type communication category.

In [12]:
# Communication dollars by type and each type's share of the communication total
comm = campaign[campaign["category"].isin(COMM)]
comm_mix = comm.pivot_table(index="category", columns="who", values="amount", aggfunc="sum", fill_value=0)
comm_mix = comm_mix.reindex(index=COMM, columns=WHO, fill_value=0)
comm_total = comm_mix.sum()
comm_long = comm_mix.reset_index().melt(id_vars="category", var_name="candidate", value_name="dollars")
comm_long["pct_of_communication"] = comm_long["dollars"] / comm_long["candidate"].map(comm_total) * 100
comm_long.to_csv(OUT + "05_communication_mix.csv", index=False)
print("Communication totals ($M):", (comm_total / 1e6).round(2).to_dict())
comm_long

Communication totals ($M): {'Brown': 80.22, 'Moreno': 20.57}


,category,candidate,dollars,pct_of_communication
0,"TV, radio & other media",Brown,"48,431,013.77",60.37
1,Digital advertising,Brown,"24,294,063.05",30.28
2,Direct mail & printing,Brown,"4,484,492.48",5.59
3,Texting & phone,Brown,"2,933,060.27",3.66
4,Canvassing & field,Brown,"79,950.00",0.10
5,"TV, radio & other media",Moreno,"16,592,280.04",80.65
6,Digital advertising,Moreno,"1,870,906.38",9.09
7,Direct mail & printing,Moreno,"1,523,814.23",7.41
8,Texting & phone,Moreno,"561,475.30",2.73
9,Canvassing & field,Moreno,"23,720.12",0.12


In [13]:
# Chart 5: types of communication
fig = go.Figure()
centres = {"Brown": 0.235, "Moreno": 0.765}
for i, w in enumerate(WHO):
    part = comm_long[comm_long["candidate"] == w]
    slice_text = [f"{p:.1f}%<br>{money_m(d)}" if p >= 5 else "" for p, d in zip(part["pct_of_communication"], part["dollars"])]
    hover = [f"<b>{c}</b><br>{w}: {money_m(d)} ({p:.1f}% of his communication spending)"
             for c, d, p in zip(part["category"], part["dollars"], part["pct_of_communication"])]
    fig.add_trace(go.Pie(labels=part["category"], values=part["dollars"], hole=0.55, sort=False,
                         domain=dict(x=[centres[w] - 0.215, centres[w] + 0.215], y=[0.14, 0.86]),
                         marker=dict(colors=[COMM_COLORS[c] for c in part["category"]]),
                         text=slice_text, textinfo="text", textposition="inside", insidetextorientation="horizontal",
                         hovertext=hover, hoverinfo="text", showlegend=(i == 0), name=w))
    small = part[part["pct_of_communication"] < 5]
    small_text = "<br>".join(f"{c}: {p:.1f}% ({money_m(d)})" for c, p, d in zip(small["category"], small["pct_of_communication"], small["dollars"]))
    fig.add_annotation(text=f"<b style='color:{COLORS[w]}'>{w}</b>", x=centres[w], y=0.97, xref="paper", yref="paper",
                       xanchor="center", yanchor="middle", showarrow=False, font=dict(size=20))
    fig.add_annotation(text=f"<b>{money_m(comm_total[w])}</b><br>communication", x=centres[w], y=0.5, xref="paper", yref="paper",
                       xanchor="center", yanchor="middle", showarrow=False, font=dict(size=15))
    fig.add_annotation(text="Smaller slices:<br>" + small_text, x=centres[w], y=0.12, xref="paper", yref="paper",
                       xanchor="center", yanchor="top", showarrow=False, font=dict(size=11, color="#444444"))
finish(fig, "05_communication_donuts.html", "How each campaign communicated with voters", SCOPE, height=660, top=150)

## Metric 6. Communication spending by quarter
**What it measures:** voter-communication dollars in each quarter, stacked by communication type. Use the dropdown to switch between Brown and Moreno.

In [14]:
# Communication dollars by quarter and type
QUARTERS = sorted(campaign["quarter"].unique())
rows = []
for w in WHO:
    part = comm[comm["who"] == w]
    pv = part.pivot_table(index="quarter", columns="category", values="amount", aggfunc="sum", fill_value=0)
    pv = pv.reindex(index=QUARTERS, columns=COMM, fill_value=0)
    for q in QUARTERS:
        for c in COMM:
            rows.append({"candidate": w, "quarter": q, "category": c, "dollars": pv.loc[q, c]})
comm_by_quarter = pd.DataFrame(rows)
comm_by_quarter.to_csv(OUT + "06_communication_by_quarter.csv", index=False)
comm_by_quarter.pivot_table(index="quarter", columns="candidate", values="dollars", aggfunc="sum") / 1e6

candidate,Brown,Moreno
quarter,,
2023Q1,0.55,0.00
2023Q2,0.56,0.31
2023Q3,0.96,0.25
2023Q4,0.81,3.18
2024Q1,8.50,3.25
2024Q2,15.84,0.85
2024Q3,32.99,6.73
2024Q4,20.00,6.00


In [15]:
# Chart 6: communication spending by quarter
fig = go.Figure()
for w in WHO:
    for c in COMM:
        part = comm_by_quarter[(comm_by_quarter["candidate"] == w) & (comm_by_quarter["category"] == c)]
        hover = [f"<b>{w}, {q}</b><br>{c}: {money_m(d)}" for q, d in zip(part["quarter"], part["dollars"])]
        fig.add_trace(go.Bar(x=part["quarter"], y=part["dollars"] / 1e6, name=c, marker_color=COMM_COLORS[c],
                             hovertext=hover, hoverinfo="text", visible=(w == "Brown")))
fig.update_layout(barmode="stack", updatemenus=[dict(
    type="dropdown", x=0, xanchor="left", y=1.02, yanchor="bottom",
    buttons=[dict(label="Brown", method="update", args=[{"visible": [True] * 5 + [False] * 5}]),
             dict(label="Moreno", method="update", args=[{"visible": [False] * 5 + [True] * 5}])])])
fig.update_xaxes(title_text="Quarter")
fig.update_yaxes(title_text="Communication spending ($M)")
finish(fig, "06_communication_by_quarter.html", "Communication spending by quarter (choose a candidate)", SCOPE, height=620)

## Metric 7. Biggest vendors
**What it measures:** the ten payees that received the most money from each campaign. Use the dropdown to switch candidate. Hover shows each vendor's share of that candidate's total spending. Names are the cleaned `payee` column.

In [16]:
# Top 10 vendors for each candidate
vendor_rows = []
for w in WHO:
    g = campaign[campaign["who"] == w]
    v = g.groupby("payee")["amount"].agg(paid="sum", payments="count").sort_values("paid", ascending=False).head(10)
    v = v.reset_index()
    v["candidate"] = w
    v["rank"] = range(1, len(v) + 1)
    v["share_of_own_spending_pct"] = v["paid"] / total[w] * 100
    vendor_rows.append(v)
top_vendors = pd.concat(vendor_rows)[["candidate", "rank", "payee", "paid", "payments", "share_of_own_spending_pct"]]
top_vendors.to_csv(OUT + "07_top_vendors.csv", index=False)
top_vendors

,candidate,rank,payee,paid,payments,share_of_own_spending_pct
0,Brown,1,GRASSROOTS MEDIA,"46,901,324.04",42,48.12
1,Brown,2,WAVELENGTH STRATEGY,"16,167,430.70",39,16.59
2,Brown,3,MISSION WIRED,"11,590,965.43",75,11.89
3,Brown,4,PRIMEPAY,"5,577,868.83",166,5.72
4,Brown,5,WILKE COMMUNICATIONS,"3,988,388.00",50,4.09
5,Brown,6,ACTBLUE TECHNICAL SERVICES,"2,542,785.19",193,2.61
6,Brown,7,SWITCHBOARD COMMUNICATIONS,"1,980,289.70",21,2.03
7,Brown,8,MAGNUS PEARSON MEDIA,"965,397.41",17,0.99
8,Brown,9,STRIVE MESSAGING,"925,721.41",24,0.95
9,Brown,10,AMERICAN EXPRESS,"592,822.43",16,0.61


In [17]:
# Chart 7: top vendors
fig = go.Figure()
for w in WHO:
    part = top_vendors[top_vendors["candidate"] == w]
    hover = [f"<b>{p.title()}</b><br>Paid by {w}: {money_m(a)}<br>{s:.1f}% of {w}'s spending<br>{n:,} payments"
             for p, a, s, n in zip(part["payee"], part["paid"], part["share_of_own_spending_pct"], part["payments"])]
    fig.add_trace(go.Bar(y=part["payee"].str.title(), x=part["paid"] / 1e6, orientation="h", name=w,
                         marker_color=COLORS[w], hovertext=hover, hoverinfo="text", visible=(w == "Brown")))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title_text="Total paid ($M)")
fig.update_layout(updatemenus=[dict(
    type="dropdown", x=0, xanchor="left", y=1.02, yanchor="bottom",
    buttons=[dict(label="Brown", method="update", args=[{"visible": [True, False]}]),
             dict(label="Moreno", method="update", args=[{"visible": [False, True]}])])])
finish(fig, "07_top_vendors.html", "Top 10 vendors paid by each campaign (choose a candidate)", SCOPE, height=620)

## Metric 8. Payment size
**What it measures:** (a) every payment as a dot, by date and amount; (b) how payments fall into five size bands, by number of payments and by dollars.

In [18]:
# One row per payment
payments = campaign[["who", "date", "payee", "purpose", "amount"]].sort_values("date").copy()
payments = payments.rename(columns={"who": "candidate"})
payments.to_csv(OUT + "08a_payments.csv", index=False)
print(len(payments), "payments")

4700 payments


In [19]:
# Chart 8a: every payment as a dot
fig = go.Figure()
for w in WHO:
    part = payments[payments["candidate"] == w]
    hover = [f"<b>{p.title()}</b><br>{purpose}<br>{d:%b %d, %Y}<br>${a:,.2f}"
             for p, purpose, d, a in zip(part["payee"], part["purpose"], part["date"], part["amount"])]
    fig.add_trace(go.Scatter(x=part["date"], y=part["amount"], mode="markers", name=w,
                             marker=dict(size=6, color=COLORS[w], opacity=0.6), hovertext=hover, hoverinfo="text"))
fig.update_xaxes(title_text="Payment date")
fig.update_yaxes(title_text="Payment amount ($)", tickprefix="$", tickformat=",", range=[-150000, 3000000])
finish(fig, "08a_payment_scatter.html", "Every payment: when and how big", SCOPE, height=680, top=170, legend_y=1.1)

In [20]:
# Payments grouped into five size bands
BANDS = ["Under $1K", "$1K–$10K", "$10K–$100K", "$100K–$1M", "Over $1M"]

# "Over $100K" and "over $1M" mean strictly greater than
def size_band(a):
    if a < 1000:
        return BANDS[0]
    if a < 10000:
        return BANDS[1]
    if a <= 100000:
        return BANDS[2]
    if a <= 1000000:
        return BANDS[3]
    return BANDS[4]

campaign["band"] = campaign["amount"].apply(size_band)
band_table = campaign.groupby(["who", "band"])["amount"].agg(payments="count", dollars="sum")
full_index = pd.MultiIndex.from_product([WHO, BANDS], names=["who", "band"])
band_table = band_table.reindex(full_index, fill_value=0).reset_index().rename(columns={"who": "candidate"})
band_table["pct_of_payments"] = band_table["payments"] / band_table["candidate"].map(campaign.groupby("who").size()) * 100
band_table["pct_of_dollars"] = band_table["dollars"] / band_table["candidate"].map(total) * 100
band_table.to_csv(OUT + "08b_size_bands.csv", index=False)
band_table

,candidate,band,payments,dollars,pct_of_payments,pct_of_dollars
0,Brown,Under $1K,1261,"339,943.26",50.10,0.35
1,Brown,$1K–$10K,629,"2,497,540.22",24.99,2.56
2,Brown,$10K–$100K,484,"15,100,239.50",19.23,15.49
3,Brown,$100K–$1M,113,"35,868,113.71",4.49,36.80
4,Brown,Over $1M,30,"43,654,747.86",1.19,44.79
5,Moreno,Under $1K,1257,"268,723.54",57.58,1.04
6,Moreno,$1K–$10K,637,"2,071,539.20",29.18,8.02
7,Moreno,$10K–$100K,254,"6,561,445.00",11.64,25.41
8,Moreno,$100K–$1M,29,"8,885,015.19",1.33,34.41
9,Moreno,Over $1M,6,"8,034,197.63",0.27,31.12


In [21]:
# Chart 8b: payment size bands
BAND_COLORS = ["#dadaeb", "#bcbddc", "#9e9ac8", "#756bb1", "#54278f"]
fig = go.Figure()
for mode in ["payments", "dollars"]:
    for band, colour in zip(BANDS, BAND_COLORS):
        part = band_table[band_table["band"] == band].set_index("candidate").loc[WHO]
        values = part["pct_of_payments"] if mode == "payments" else part["pct_of_dollars"]
        hover = [f"<b>{w}: {band}</b><br>{int(part.loc[w, 'payments']):,} payments ({part.loc[w, 'pct_of_payments']:.1f}% of his payments)"
                 f"<br>{money_m(part.loc[w, 'dollars'])} ({part.loc[w, 'pct_of_dollars']:.1f}% of his dollars)" for w in WHO]
        fig.add_trace(go.Bar(y=WHO, x=values, orientation="h", name=band, marker_color=colour,
                             text=[f"{v:.1f}%" if v >= 4 else "" for v in values], textposition="inside",
                             hovertext=hover, hoverinfo="text", visible=(mode == "payments")))
fig.update_layout(barmode="stack", updatemenus=[dict(
    type="buttons", direction="right", x=0, xanchor="left", y=1.02, yanchor="bottom",
    buttons=[dict(label="% of payments (count)", method="update",
                  args=[{"visible": [True] * 5 + [False] * 5}, {"xaxis.title.text": "Share of the candidate's payments (%)"}]),
             dict(label="% of dollars", method="update",
                  args=[{"visible": [False] * 5 + [True] * 5}, {"xaxis.title.text": "Share of the candidate's dollars (%)"}])])])
fig.update_xaxes(title_text="Share of the candidate's payments (%)", range=[0, 100])
fig.update_yaxes(autorange="reversed")
finish(fig, "08b_payment_size_bands.html", "Payment size: many small payments, few huge ones", SCOPE, height=560)

## Metric 9. What each committee spent on
**What it measures:** each committee's spending split into three types: voter communication, fundraising costs, and staff, overhead and other. This is the **only** metric that shows the **affiliated** committees (joint fundraising / victory committees). They are labelled "affiliated" and are **not** part of any other number in this notebook. Use the buttons to switch between $ and % of the committee's own spending.

In [22]:
# Spending types for every committee, campaign and affiliated
TYPES = ["Voter communication", "Fundraising costs", "Staff, overhead and other"]
TYPE_COLORS = dict(zip(TYPES, ["#6a51a3", "#e6a817", "#9e9e9e"]))
both = df.copy()
both["spending_type"] = "Staff, overhead and other"
both.loc[both["category"].isin(COMM), "spending_type"] = "Voter communication"
both.loc[both["category"] == "Fundraising costs", "spending_type"] = "Fundraising costs"

pv = both.pivot_table(index=["who", "group", "committee"], columns="spending_type", values="amount", aggfunc="sum", fill_value=0)
pv = pv.reindex(columns=TYPES, fill_value=0)
pv["total"] = pv.sum(axis=1)
by_committee = pv.reset_index()
by_committee["order_who"] = by_committee["who"].map({"Brown": 0, "Moreno": 1})
by_committee["order_group"] = by_committee["group"].map({"campaign": 0, "affiliated": 1})
by_committee = by_committee.sort_values(["order_who", "order_group", "total"], ascending=[True, True, False])
by_committee = by_committee.drop(columns=["order_who", "order_group"]).reset_index(drop=True)
by_committee["committee_label"] = by_committee["who"] + " – " + by_committee["committee"] + " (" + by_committee["group"] + ")"
for t in TYPES:
    by_committee[t + " (%)"] = by_committee[t] / by_committee["total"] * 100
by_committee = by_committee.rename(columns={"who": "candidate"})
by_committee.to_csv(OUT + "09_committee_spending.csv", index=False)
by_committee[["committee_label"] + TYPES + ["total"]]

spending_type,committee_label,Voter communication,Fundraising costs,"Staff, overhead and other",total
0,Brown – Friends of Sherrod Brown (campaign),"80,222,579.57","4,356,828.10","12,881,176.88","97,460,584.55"
1,Brown – Ohio Grassroots Victory Fund (affiliated),"51,739.55","541,202.17","2,328,452.74","2,921,394.46"
2,Brown – Senate Victory for Sherrod Brown 2024 ...,0.00,0.00,"60,599.39","60,599.39"
3,Brown – Victory Now for Sherrod Brown (affilia...,0.00,0.00,"22,732.82","22,732.82"
4,Brown – Canary Fund (affiliated),0.00,0.00,"1,943.91","1,943.91"
5,Moreno – Bernie Moreno for Senate (campaign),"20,572,196.07","1,131,935.57","4,101,561.64","25,805,693.28"
6,Moreno – Moreno for OH Senate Republican Nomin...,0.00,0.00,"13,190.77","13,190.77"
7,Moreno – Bernie Moreno for Ohio (campaign),0.00,0.00,"2,036.51","2,036.51"
8,Moreno – Team Moreno (affiliated),"2,165,190.13","1,097,357.56","1,040,362.71","4,302,910.40"


In [23]:
# Chart 9: what each committee spent on
fig = go.Figure()
for mode in ["dollars", "percent"]:
    for t in TYPES:
        if mode == "dollars":
            values = by_committee[t] / 1e6
        else:
            values = by_committee[t + " (%)"]
        hover = [f"<b>{r['committee_label']}</b><br>{t}: {money_m(r[t])} ({r[t + ' (%)']:.1f}% of this committee's spending)"
                 for _, r in by_committee.iterrows()]
        fig.add_trace(go.Bar(y=by_committee["committee_label"], x=values, orientation="h", name=t,
                             marker_color=TYPE_COLORS[t], hovertext=hover, hoverinfo="text",
                             visible=(mode == "dollars"), showlegend=(mode == "dollars")))
fig.update_layout(barmode="stack", updatemenus=[dict(
    type="buttons", direction="right", x=0, xanchor="left", y=1.02, yanchor="bottom",
    buttons=[dict(label="$ spent", method="update",
                  args=[{"visible": [True] * 3 + [False] * 3, "showlegend": [True] * 3 + [False] * 3}, {"xaxis.title.text": "Spent ($M)"}]),
             dict(label="% of committee's spending", method="update",
                  args=[{"visible": [False] * 3 + [True] * 3, "showlegend": [False] * 3 + [True] * 3}, {"xaxis.title.text": "Share of the committee's own spending (%)"}])])])
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title_text="Spent ($M)")
finish(fig, "09_committees.html", "What each committee spent on (campaign vs affiliated)",
       "All committees: 'campaign' = candidate's own, 'affiliated' = joint fundraising / victory committees, 2023–2024", height=680, top=150, legend_right=True)

## Metric 10. Staffing and fundraising costs (table only)
**What it measures:** what each campaign spent on payroll, fundraising costs, payment-processing fees and overhead, in dollars and as a % of its own spending.

**Caution on payroll:** Moreno's payroll shows only about $0.10M against Brown's $5.95M. This is **probably a reporting or categorisation difference** (for example, staff paid through a vendor or described in another way), so it is **not** a finding about who has bigger staff.

In [24]:
# Payroll, fundraising, processing and operations costs
lines = ["Payroll & benefits", "Fundraising costs", "Processing & bank fees", "Operations & overhead"]
staff = category_table.set_index("category").loc[lines]
staffing = pd.DataFrame({"line": lines,
                         "Brown ($)": staff["Brown"].values, "Brown (% of own spending)": staff["Brown_pct"].values,
                         "Moreno ($)": staff["Moreno"].values, "Moreno (% of own spending)": staff["Moreno_pct"].values})
staffing["note"] = ["Probably a reporting difference; do not treat as a finding", "", "Card and payment-processing fees", "Rent, software, office costs"]
staffing.to_csv(OUT + "10_staffing_fundraising.csv", index=False)
staffing

,line,Brown ($),Brown (% of own spending),Moreno ($),Moreno (% of own spending),note
0,Payroll & benefits,"5,951,014.68",6.11,"101,207.25",0.39,Probably a reporting difference; do not treat ...
1,Fundraising costs,"4,356,828.10",4.47,"1,131,935.57",4.38,
2,Processing & bank fees,"2,689,479.63",2.76,"363,734.88",1.41,Card and payment-processing fees
3,Operations & overhead,"1,233,279.54",1.27,"156,813.82",0.61,"Rent, software, office costs"


**What this shows:** fundraising costs are a similar share of each candidate's spending (4.5% for Brown, 4.4% for Moreno). Brown's processing fees are a bigger share (2.8% vs 1.4%). Payroll differs sharply (6.1% vs 0.4%) but see the caution above.

## Sensitivity check: what if we drop the `possible_repeat` payments?
`possible_repeat` flags look-alike payments (same committee, date, payee and amount). They are kept in all results above. Here we drop them to see how much the headline numbers move.

In [25]:
# Headline numbers with and without possible repeat rows
def headline(frame):
    """The headline numbers of this notebook for one version of the data."""
    out = {}
    for w in WHO:
        g = frame[frame["who"] == w]
        t = g["amount"].sum()
        vendors = g.groupby("payee")["amount"].sum().sort_values(ascending=False)
        out[w + " total spent ($M)"] = t / 1e6
        out[w + " share on voter communication (%)"] = g[g["category"].isin(COMM)]["amount"].sum() / t * 100
        out[w + " median payment ($)"] = g["amount"].median()
        out[w + " payments over $100K"] = (g["amount"] > 100000).sum()
        out[w + " payments over $1M"] = (g["amount"] > 1000000).sum()
        out[w + " biggest vendor's share (%)"] = vendors.iloc[0] / t * 100
    out["Gap, Brown minus Moreno ($M)"] = out["Brown total spent ($M)"] - out["Moreno total spent ($M)"]
    return pd.Series(out)

sensitivity = pd.DataFrame({"with all payments": headline(campaign),
                            "without possible_repeat": headline(campaign[campaign["possible_repeat"] == False])})
sensitivity["change"] = sensitivity["without possible_repeat"] - sensitivity["with all payments"]
sensitivity["change (%)"] = sensitivity["change"] / sensitivity["with all payments"] * 100
sensitivity = sensitivity.reset_index().rename(columns={"index": "measure"})
sensitivity.to_csv(OUT + "11_sensitivity.csv", index=False)
print(campaign["possible_repeat"].sum(), "possible_repeat payments in campaign scope")
sensitivity

10 possible_repeat payments in campaign scope


,measure,with all payments,without possible_repeat,change,change (%)
0,Brown total spent ($M),97.46,97.42,-0.04,-0.04
1,Brown share on voter communication (%),82.31,82.32,0.01,0.01
2,Brown median payment ($),996.00,993.21,-2.79,-0.28
3,Brown payments over $100K,143.00,143.00,0.00,0.00
4,Brown payments over $1M,30.00,30.00,0.00,0.00
5,Brown biggest vendor's share (%),48.12,48.14,0.02,0.04
6,Moreno total spent ($M),25.82,25.82,-0.00,-0.00
7,Moreno share on voter communication (%),79.67,79.67,0.00,0.00
8,Moreno median payment ($),548.47,552.19,3.72,0.68
9,Moreno payments over $100K,35.00,35.00,0.00,0.00


## Validation
Every check compares a result with a reference number, or compares a chart with its saved table. All checks must pass.

In [26]:
# Checks on every result and chart
checks = []

def check(name, expected, actual, tol=0.01):
    """Record one check: PASS when |expected - actual| <= tol."""
    ok = abs(float(expected) - float(actual)) <= tol
    checks.append({"check": name, "expected": expected, "actual": actual, "pass_fail": "PASS" if ok else "FAIL"})

def plotted_equals_table(name, fig, trace_numbers, table_values, scale=1.0):
    """Largest difference between what the chart plots and the saved table (should be 0)."""
    plotted = np.concatenate([np.array(fig.data[i].x, dtype=float) for i in trace_numbers])
    expected = np.array(table_values, dtype=float) / scale
    check(name, 0, np.abs(plotted - expected).max(), tol=1e-6)

# --- 1. Reference totals (campaign scope) ---
check("Brown campaign total ($)", 97460584.55, total["Brown"], 0.005)
check("Moreno campaign total ($)", 25820920.56, total["Moreno"], 0.005)
check("Total gap ($)", 71639663.99, total["Brown"] - total["Moreno"], 0.005)
check("Brown total with affiliated committees ($)", 100467255.13, df[df["who"] == "Brown"]["amount"].sum(), 0.005)
check("Moreno total with affiliated committees ($)", 30123830.96, df[df["who"] == "Moreno"]["amount"].sum(), 0.005)

# --- 2. Category sums and gap by category ---
for w in WHO:
    check(f"{w}: category dollars add up to total ($)", total[w], category_table[w].sum(), 0.005)
    check(f"{w}: category % add up to 100", 100, category_table[w + "_pct"].sum(), 1e-6)
check("All 16 categories present", 16, category_table["category"].nunique(), 0)
check("No missing categories in the data", 0, campaign["category"].isna().sum(), 0)
check("No missing dates in the data", 0, campaign["date"].isna().sum(), 0)
check("All 8 quarters present", 8, len(QUARTERS), 0)
gap_ref = {"TV, radio & other media": 31.84, "Digital advertising": 22.42, "Payroll & benefits": 5.85,
           "Fundraising costs": 3.23, "Direct mail & printing": 2.96, "Texting & phone": 2.37, "Processing & bank fees": 2.33}
for cname, ref in gap_ref.items():
    check(f"Gap in $M: {cname}", ref, gap.set_index("category").loc[cname, "gap"] / 1e6, 0.015)
check("Gap bars add up to the total gap ($)", total["Brown"] - total["Moreno"], gap["gap"].sum(), 0.005)

# --- 3. Communication ---
check("Brown communication ($M)", 80.22, comm_total["Brown"] / 1e6, 0.005)
check("Moreno communication ($M)", 20.57, comm_total["Moreno"] / 1e6, 0.005)
check("Brown communication share (%)", 82.3, stats.loc["communication_share_pct", "Brown"], 0.05)
check("Moreno communication share (%)", 79.7, stats.loc["communication_share_pct", "Moreno"], 0.05)
mix_ref = {"Brown": [60.4, 30.3, 5.6, 3.7], "Moreno": [80.7, 9.1, 7.4, 2.7]}
for w in WHO:
    pct = comm_long[comm_long["candidate"] == w].set_index("category")["pct_of_communication"]
    for cname, ref in zip(COMM[:4], mix_ref[w]):
        check(f"{w} channel mix: {cname} (%)", ref, pct[cname], 0.05)
    check(f"{w}: donut slices add up to 100%", 100, pct.sum(), 1e-6)
    check(f"{w}: donut slices equal communication total ($)", comm_total[w], comm_long[comm_long["candidate"] == w]["dollars"].sum(), 0.005)
    check(f"{w}: quarterly communication adds up to communication total ($)", comm_total[w],
          comm_by_quarter[comm_by_quarter["candidate"] == w]["dollars"].sum(), 0.005)
for i, w in enumerate(WHO):
    check(f"{w}: donut chart values equal communication total ($)", comm_total[w], sum(FIGS["05_communication_donuts.html"].data[i].values), 0.005)

# --- 4. Vendors ---
tv = top_vendors.set_index(["candidate", "rank"])
check("Grassroots Media paid by Brown ($M)", 46.9, tv.loc[("Brown", 1), "paid"] / 1e6, 0.05)
check("Grassroots Media share of Brown (%)", 48.1, tv.loc[("Brown", 1), "share_of_own_spending_pct"], 0.05)
check("Strategic Media Placement paid by Moreno ($M)", 16.1, tv.loc[("Moreno", 1), "paid"] / 1e6, 0.05)
check("Strategic Media Placement share of Moreno (%)", 62.2, tv.loc[("Moreno", 1), "share_of_own_spending_pct"], 0.05)
for w in WHO:
    top10 = top_vendors[top_vendors["candidate"] == w]["paid"].sum()
    check(f"{w}: top-10 vendors paid is not more than total (1 = yes)", 1, int(top10 <= total[w]), 0)

# --- 5. Payment size ---
check("Brown payments over $1M", 30, (campaign[campaign["who"] == "Brown"]["amount"] > 1000000).sum(), 0)
check("Moreno payments over $1M", 6, (campaign[campaign["who"] == "Moreno"]["amount"] > 1000000).sum(), 0)
check("Brown payments over $1M, % of his spending", 44.8, band_table[(band_table["candidate"] == "Brown") & (band_table["band"] == "Over $1M")]["pct_of_dollars"].iloc[0], 0.05)
check("Moreno payments over $1M, % of his spending", 31.1, band_table[(band_table["candidate"] == "Moreno") & (band_table["band"] == "Over $1M")]["pct_of_dollars"].iloc[0], 0.05)
check("Brown payments over $100K", 143, (campaign[campaign["who"] == "Brown"]["amount"] > 100000).sum(), 0)
check("Moreno payments over $100K", 35, (campaign[campaign["who"] == "Moreno"]["amount"] > 100000).sum(), 0)
check("Brown median payment ($)", 996, stats.loc["median_payment", "Brown"], 0.5)
check("Moreno median payment ($)", 548, stats.loc["median_payment", "Moreno"], 0.5)
for w in WHO:
    part = band_table[band_table["candidate"] == w]
    n_payments = (campaign["who"] == w).sum()
    check(f"{w}: size-band counts add up to payment count", n_payments, part["payments"].sum(), 0)
    check(f"{w}: size-band dollars add up to total ($)", total[w], part["dollars"].sum(), 0.005)
check("Scatter has one dot per payment", len(campaign), sum(len(t.x) for t in FIGS["08a_payment_scatter.html"].data), 0)

# --- 6. Committees ---
for w, ref in [("Brown", 100467255.13), ("Moreno", 30123830.96)]:
    part = by_committee[by_committee["candidate"] == w]
    check(f"{w}: committee table = campaign + affiliated total ($)", ref, part["total"].sum(), 0.005)
    check(f"{w}: campaign committees in committee table = campaign total ($)", total[w], part[part["group"] == "campaign"]["total"].sum(), 0.005)
    check(f"{w}: affiliated committees in committee table = affiliated total ($)", affiliated[affiliated["who"] == w]["amount"].sum(), part[part["group"] == "affiliated"]["total"].sum(), 0.005)
fig9 = FIGS["09_committees.html"]
check("Committee chart $ bars add up to campaign + affiliated total ($)", df["amount"].sum(),
      sum(sum(fig9.data[i].x) for i in range(3)) * 1e6, 0.5)

# --- 7. Every chart shows exactly what is in its saved table ---
f2, f4 = FIGS["02_category_spending.html"], FIGS["04_brown_advantage.html"]
plotted_equals_table("Chart 2 Brown $M = table", f2, [0], category_table["Brown"], 1e6)
plotted_equals_table("Chart 2 Moreno $M = table", f2, [1], category_table["Moreno"], 1e6)
plotted_equals_table("Chart 2 Brown % = table", f2, [2], category_table["Brown_pct"])
plotted_equals_table("Chart 2 Moreno % = table", f2, [3], category_table["Moreno_pct"])
plotted_equals_table("Chart 4 gap bars = table", f4, [0, 1], pd.concat([gap[gap["gap"] >= 0]["gap"], gap[gap["gap"] < 0]["gap"]]), 1e6)
f3 = FIGS["03_category_scatter.html"]
check("Chart 3 left panel has all 16 categories", 16, len(f3.data[0].x) + len(f3.data[1].x), 0)
f6 = FIGS["06_communication_by_quarter.html"]
check("Chart 6 plotted $ = table ($)", comm_by_quarter["dollars"].sum(), sum(sum(t.y) for t in f6.data) * 1e6, 0.5)
f7 = FIGS["07_top_vendors.html"]
check("Chart 7 plotted $ = table ($)", top_vendors["paid"].sum(), sum(sum(t.x) for t in f7.data) * 1e6, 0.5)
f8 = FIGS["08b_payment_size_bands.html"]
check("Chart 8b plotted % of payments = table", band_table["pct_of_payments"].sum(), sum(sum(f8.data[i].x) for i in range(5)), 1e-6)
check("Chart 8b plotted % of dollars = table", band_table["pct_of_dollars"].sum(), sum(sum(f8.data[i].x) for i in range(5, 10)), 1e-6)
check("Chart 1 scorecard cards: 17 cards", 17, len(FIGS["01_scorecard.html"].data), 0)

# --- 8. Every chart file exists ---
expected_charts = ["01_scorecard.html", "02_category_spending.html", "03_category_scatter.html", "04_brown_advantage.html",
                   "05_communication_donuts.html", "06_communication_by_quarter.html", "07_top_vendors.html",
                   "08a_payment_scatter.html", "08b_payment_size_bands.html", "09_committees.html"]
for name in expected_charts:
    check(f"Chart file exists: {name}", 1, int(os.path.exists(OUT + name)), 0)

validation = pd.DataFrame(checks)
validation.to_csv(OUT + "validation_checks.csv", index=False)
print(validation["pass_fail"].value_counts().to_dict())
assert (validation["pass_fail"] == "PASS").all(), "Some checks failed: look at the FAIL rows below"
validation

{'PASS': 88}


,check,expected,actual,pass_fail
0,Brown campaign total ($),"97,460,584.55","97,460,584.55",PASS
1,Moreno campaign total ($),"25,820,920.56","25,820,920.56",PASS
2,Total gap ($),"71,639,663.99","71,639,663.99",PASS
3,Brown total with affiliated committees ($),"100,467,255.13","100,467,255.13",PASS
4,Moreno total with affiliated committees ($),"30,123,830.96","30,123,830.96",PASS
...,...,...,...,...
83,Chart file exists: 06_communication_by_quarter...,1.00,1.00,PASS
84,Chart file exists: 07_top_vendors.html,1.00,1.00,PASS
85,Chart file exists: 08a_payment_scatter.html,1.00,1.00,PASS
86,Chart file exists: 08b_payment_size_bands.html,1.00,1.00,PASS
